In [1]:
from __future__ import annotations
import os
import json
import importlib
import argparse
from functools import partial

import random
import numpy as np

import torch


from vllm import LLM, SamplingParams, PoolingParams

from sal.config import Config

from core import test_nphases_mcts_search_v55
from core.reward_models import RLHFFlow
from utils.load_data import load_data_prm800k

# from core.llm_engine import rm_engine
# from core.llms import rm_generate
# import logging
# logging.basicConfig(format='%(message)s', level=logging.ERROR)

INFO 08-19 16:35:08 [__init__.py:244] Automatically detected platform cuda.


In [2]:
if torch.cuda.is_available():
    GPUS = os.environ.get('CUDA_VISIBLE_DEVICES', "0").split(',')
    print(GPUS)
else:
    print("CUDA is not available.")

['0']


In [3]:
# base_dir
base_dir = '/groups/kjun/tnn/datasets/'

# dataset path
data_dir = base_dir + "/prm800k/math_splits"

# llm and prm path
# llm_dir = base_dir + "/Llama-3.2-1B-Instruct-GGUF/Llama-3.2-1B-Instruct.Q4_K_M.gguf"
# prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-GGUF/Llama3.1-8B-PRM-Deepseek-Data.Q4_K_M.gguf"

llm_dir = base_dir + "/Llama-3.2-1B-Instruct"
prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data"
# prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-Modified"

In [4]:
llm_total_gpu = 0.4
llm_gpu_memory_utilization = 0.2
# llm_vllm = LLM(
#     model = llm_dir,
#     tensor_parallel_size=1,
#     gpu_memory_utilization = 0.7,  # Utilize 50% of GPU memory
#     # enable_prefix_caching=True,  # V100 doesn't support enable_prefix_caching 
#     # enable_chunked_prefill=False, # and enable_chunked_prefill
#     max_model_len = 5000,
#     dtype = "float16",
#     seed = config.seed)

llm_vllm = LLM(
    model=llm_dir, 
    tensor_parallel_size=1, 
    # trust_remote_code=True,
    swap_space=16,
    max_model_len=5000,
    gpu_memory_utilization=llm_gpu_memory_utilization,
    enforce_eager=True,
    distributed_executor_backend=None,
    dtype="float16",
    seed=0,
)

print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))



INFO 08-19 16:35:28 [config.py:823] This model supports multiple tasks: {'generate', 'classify', 'embed', 'score', 'reward'}. Defaulting to 'generate'.
WARNING 08-19 16:35:28 [config.py:3271] Casting torch.bfloat16 to torch.float16.
WARNING 08-19 16:35:28 [arg_utils.py:1642] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
WARNING 08-19 16:35:28 [cuda.py:91] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 08-19 16:35:28 [llm_engine.py:230] Initializing a V0 LLM engine (v0.9.1) with config: model='/groups/kjun/tnn/datasets//Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='/groups/kjun/tnn/datasets//Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=5000, download_dir=None, load_format=LoadFormat.AUTO, tensor_par

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 08-19 16:35:31 [default_loader.py:272] Loading weights took 1.27 seconds
INFO 08-19 16:35:32 [model_runner.py:1203] Model loading took 2.3185 GiB and 1.388482 seconds
INFO 08-19 16:35:33 [worker.py:294] Memory profiling takes 0.51 seconds
INFO 08-19 16:35:33 [worker.py:294] the current vLLM instance can use total_gpu_memory (31.73GiB) x gpu_memory_utilization (0.20) = 6.35GiB
INFO 08-19 16:35:33 [worker.py:294] model weights take 2.32GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 1.19GiB; the rest of the memory reserved for KV Cache is 2.75GiB.
INFO 08-19 16:35:33 [executor_base.py:113] # cuda blocks: 5631, # CPU blocks: 32768
INFO 08-19 16:35:33 [executor_base.py:118] Maximum concurrency for 5000 tokens per request: 18.02x
INFO 08-19 16:35:39 [llm_engine.py:428] init engine (profile, create kv cache, warmup model) took 7.14 seconds
#--- memory: 5.07647705078125


In [5]:
llm_vllm_embeds = LLM(
    model=llm_dir, 
    tensor_parallel_size=1, 
    # trust_remote_code=True,
    task="embed",
    swap_space=16,
    max_model_len=5000,
    gpu_memory_utilization=llm_total_gpu-llm_gpu_memory_utilization,
    enforce_eager=True,
    distributed_executor_backend=None,
    dtype="float16",
    seed=0,
)
print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))

WARNING 08-19 16:35:39 [config.py:3271] Casting torch.bfloat16 to torch.float16.
WARNING 08-19 16:35:39 [cuda.py:91] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 08-19 16:35:39 [llm_engine.py:230] Initializing a V0 LLM engine (v0.9.1) with config: model='/groups/kjun/tnn/datasets//Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='/groups/kjun/tnn/datasets//Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=5000, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_pr

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 08-19 16:35:42 [default_loader.py:272] Loading weights took 1.26 seconds
INFO 08-19 16:35:42 [model_runner.py:1203] Model loading took 2.3029 GiB and 1.299962 seconds
#--- memory: 7.379390716552734


In [6]:
prm = RLHFFlow(model_path=prm_dir, device_map='cuda:0')
print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

#--- memory: 22.336918354034424


In [7]:
stop

NameError: name 'stop' is not defined

In [8]:
#  load data 
data_by_levels = load_data_prm800k(data_dir)

1: 43
2: 90
3: 105
4: 128
5: 134


In [41]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# general params
config = Config()
config.agg_strategy = 'last'

config.n = 4                      # number of budgets to be generated per depth
config.beam_width = 4             # number of nodes left after selection
config.lookahead = 0              # don't use it for now
config.max_depths = 40            # max depths, after reaching max_depth then terminate search 
config.sort_completed = False      
config.filter_duplicates = True   # remove any duplicates in the last list of trajs
config.seed = 0
config.date_string = "Aug 1 2025"

# mcts parameter

config.num_batches = 2
config.batch_budget = config.num_batches*config.max_depths 
config.num_phases = config.batch_budget

config.lam = 10 
config.normalize_embeds = True

config.cpuct_root = 0
config.cpuct_leaf = 0
config.cpuct = 0
config.ds_beta = 1.0
config.ds_alpha = 100000.0
config.use_ppl = True

# config.version = "v51"



In [10]:
print(config.date_string)

AttributeError: 'Config' object has no attribute 'date_string'

In [49]:
level = 4                                   # level of difficulty of questions
num_questions = len(data_by_levels[level])  # level 4 has 128 questions
num_questions = 10
num_trials = 1
print(f"num_questions = {num_questions}")
print(f"num_trials = {num_trials}")

# get batch of questions ['q1', 'q2', ...]
batch_of_questions = [data_by_levels[level][q_idx]['problem'] for q_idx in range(num_questions)]
# q_idx = 25
# batch_of_questions = [data_by_levels[level][q_idx]['problem']]

num_questions = 10
num_trials = 1


In [22]:
print(len(batch_of_questions))

1


In [13]:
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL)

In [46]:
# importlib.reload(mcts_search_v13)
# from core import test_nphases_mcts_search_v55
importlib.reload(test_nphases_mcts_search_v55)

<module 'core.test_nphases_mcts_search_v55' from '/home/u20/tnguyen9210/tnn1/LLMs/llm-reasoning-methods/core/test_nphases_mcts_search_v55.py'>

In [50]:
# print(config.cpuct)
# print(config.ds_alpha)
# print(config.ds_beta)
np.random.seed(100000+0)
random.seed(100000+0)
torch.manual_seed(100000+0)
torch.cuda.manual_seed(100000+0)

nphases_arr = []
ndepths_arr = []
for q_idx, question in enumerate(batch_of_questions):
    agent = test_nphases_mcts_search_v55.MCTS(config=config, question=question)
    agent_completions, nphases, ndepths = test_nphases_mcts_search_v55.mcts_search(question, agent, config, llm_vllm, llm_vllm_embeds, prm)
    # print(agent_completions)
    nphases_arr.append(nphases)
    ndepths_arr.append(ndepths)

nphases_mean = np.mean(nphases_arr)
nphases_std = np.std(nphases_arr, ddof=1)/np.sqrt(len(nphases_arr)) 
print(f"nphases_mean: {nphases_mean:0.1f} (\u00B1{nphases_std:0.1f})")

ndepths_mean = np.mean(ndepths_arr)
ndepths_std = np.std(ndepths_arr, ddof=1)/np.sqrt(len(ndepths_arr)) 
print(f"ndepths_mean: {ndepths_mean:0.1f} (\u00B1{ndepths_std:0.1f})")


-> p = 0
selected_node
0

-> d = 0
selected_node
0.4
batch_cnt = 1

-> d = 1
selected_node
0.4.3
batch_cnt = 2

-> d = 2
selected_node
0.4.3.1
batch_cnt = 3

-> d = 3
selected_node
0.4.3.1.4
batch_cnt = 4

-> d = 4
selected_node
0.4.3.1.4.4
batch_cnt = 5

-> d = 5
selected_node
0.4.3.1.4.4.2
batch_cnt = 6

-> d = 6
selected_node
None
batch_cnt = 7

-> d = 7

-> p = 1
selected_node
0.1

-> d = 0
selected_node
0.1.1
batch_cnt = 8

-> d = 1
selected_node
0.1.1.2
batch_cnt = 9

-> d = 2
selected_node
0.1.1.2.2
batch_cnt = 10

-> d = 3
selected_node
0.1.1.2.2.3
batch_cnt = 11

-> d = 4
selected_node
0.1.1.2.2.3.1
batch_cnt = 12

-> d = 5
selected_node
0.1.1.2.2.3.1.1
batch_cnt = 13

-> d = 6
selected_node
0.1.1.2.2.3.1.1.3
batch_cnt = 14

-> d = 7
selected_node
0.1.1.2.2.3.1.1.3.2
batch_cnt = 15

-> d = 8
selected_node
None
batch_cnt = 16

-> d = 9

-> p = 2
selected_node
0.1.1.3

-> d = 0
selected_node
0.1.1.3.3
batch_cnt = 17

-> d = 1
selected_node
0.1.1.3.3.4
batch_cnt = 18

-> d = 2
s

nphases = 16
all_depths = [7, 9, 5, 6, 5, 6, 1, 5, 7, 1, 1, 5, 5, 4, 4, 3, 5]
ndepths_mean: 4.6471 (±0.5351)


selected_node
0.1
batch_cnt = 1

-> d = 1
selected_node
0.1.3
batch_cnt = 2

-> d = 2
selected_node
0.1.3.3
batch_cnt = 3

-> d = 3
selected_node
0.1.3.3.4
batch_cnt = 4

-> d = 4
selected_node
0.1.3.3.4.2
batch_cnt = 5

-> d = 5
selected_node
0.1.3.3.4.2.2
batch_cnt = 6

-> d = 6
selected_node
0.1.3.3.4.2.2.4
batch_cnt = 7

-> d = 7
selected_node
0.1.3.3.4.2.2.4.4
batch_cnt = 8

-> d = 8
selected_node
0.1.3.3.4.2.2.4.4.4
batch_cnt = 9

-> d = 9
selected_node
0.1.3.3.4.2.2.4.4.4.2
batch_cnt = 10

-> d = 10
selected_node
0.1.3.3.4.2.2.4.4.4.2.3
batch_cnt = 11

-> d = 11
selected_node
0.1.3.3.4.2.2.4.4.4.2.3.3
batch_cnt = 12

-> d = 12
selected_node
0.1.3.3.4.2.2.4.4.4.2.3.3.1
batch_cnt = 13

-> d = 13
selected_node
0.1.3.3.4.2.2.4.4.4.2.3.3.1.4
batch_cnt = 14

-> d = 14
selected_node
0.1.3.3.4.2.2.4.4.4.2.3.3.1.4.3
batch_cnt = 15

-> d = 15
selected_node
0.1.3.3.4.2.2.4.4.4.2.3.3.1.4.3.4
batch_cnt = 16

-> d = 16
selected_node
0.1.3.3.4.2.2.4.4.4.2.3.3.1.4.3.4.2
batch_cnt = 17

-> d = 1

nphases = 6
all_depths = [20, 1, 1, 1, 14, 37, 5]
ndepths_mean: 11.2857 (±5.1210)


selected_node
0.1
batch_cnt = 1

-> d = 1
selected_node
0.1.4
batch_cnt = 2

-> d = 2
selected_node
None
batch_cnt = 3

-> d = 3

-> p = 1
selected_node
0.4

-> d = 0
selected_node
0.4.3
batch_cnt = 4

-> d = 1
selected_node
0.4.3.1
batch_cnt = 5

-> d = 2
selected_node
0.4.3.1.3
batch_cnt = 6

-> d = 3
selected_node
0.4.3.1.3.1
batch_cnt = 7

-> d = 4
selected_node
0.4.3.1.3.1.4
batch_cnt = 8

-> d = 5
selected_node
0.4.3.1.3.1.4.1
batch_cnt = 9

-> d = 6
selected_node
None
batch_cnt = 10

-> d = 7

-> p = 2
selected_node
0.3

-> d = 0
selected_node
0.3.2
batch_cnt = 11

-> d = 1
selected_node
0.3.2.3
batch_cnt = 12

-> d = 2
selected_node
None
batch_cnt = 13

-> d = 3

-> p = 3
selected_node
0.4.2

-> d = 0
selected_node
0.4.2.4
batch_cnt = 14

-> d = 1
selected_node
0.4.2.4.2
batch_cnt = 15

-> d = 2
selected_node
None
batch_cnt = 16

-> d = 3

-> p = 4
selected_node
0.2

-> d = 0
selected_node
0.2.3
batch_cnt = 17

-> d = 1
selected_node
0.2.3.1
batch_cnt = 18

-> d = 2
selected_no

...

nphases = 16
all_depths = [5, 4, 4, 3, 3, 4, 2, 2, 2, 1, 6, 2, 1, 4, 1, 0]
ndepths_mean: 2.7500 (±0.4133)


selected_node
0.4
batch_cnt = 1

-> d = 1
selected_node
0.4.1
batch_cnt = 2

-> d = 2
selected_node
0.4.1.2
batch_cnt = 3

-> d = 3
selected_node
None
batch_cnt = 4

-> d = 4

-> p = 1
selected_node
0.2

-> d = 0
selected_node
None
batch_cnt = 5

-> d = 1

-> p = 2
selected_node
0.1

-> d = 0
selected_node
0.1.2
batch_cnt = 6

-> d = 1
selected_node
0.1.2.3
batch_cnt = 7

-> d = 2
selected_node
0.1.2.3.3
batch_cnt = 8

-> d = 3
selected_node
0.1.2.3.3.3
batch_cnt = 9

-> d = 4
selected_node
0.1.2.3.3.3.3
batch_cnt = 10

-> d = 5
selected_node
0.1.2.3.3.3.3.1
batch_cnt = 11

-> d = 6
selected_node
None
batch_cnt = 12

-> d = 7

-> p = 3
selected_node
0.4.3

-> d = 0
selected_node
0.4.3.1
batch_cnt = 13

-> d = 1
selected_node
0.4.3.1.1
batch_cnt = 14

-> d = 2
selected_node
0.4.3.1.1.2
batch_cnt = 15

-> d = 3
selected_node
None
batch_cnt = 16

-> d = 4

-> p = 4
selected_node
0.3

-> d = 0
selected_node
0.3.1
batch_cnt = 17

-> d = 1
selected_node
0.3.1.2
batch_cnt = 18

-> d = 2
selec

nphases = 20
all_depths = [4, 1, 7, 4, 6, 3, 4, 1, 3, 5, 1, 1, 4, 3, 4, 2, 4, 1, 3, 0]
ndepths_mean: 3.0500 (±0.4134)


selected_node
0.2
batch_cnt = 1

-> d = 1
selected_node
0.2.3
batch_cnt = 2

-> d = 2
selected_node
0.2.3.2
batch_cnt = 3

-> d = 3
selected_node
0.2.3.2.4
batch_cnt = 4

-> d = 4
selected_node
0.2.3.2.4.2
batch_cnt = 5

-> d = 5
selected_node
0.2.3.2.4.2.4
batch_cnt = 6

-> d = 6
selected_node
0.2.3.2.4.2.4.4
batch_cnt = 7

-> d = 7
selected_node
0.2.3.2.4.2.4.4.3
batch_cnt = 8

-> d = 8
selected_node
0.2.3.2.4.2.4.4.3.1
batch_cnt = 9

-> d = 9
selected_node
0.2.3.2.4.2.4.4.3.1.2
batch_cnt = 10

-> d = 10
selected_node
0.2.3.2.4.2.4.4.3.1.2.2
batch_cnt = 11

-> d = 11
selected_node
0.2.3.2.4.2.4.4.3.1.2.2.1
batch_cnt = 12

-> d = 12
selected_node
0.2.3.2.4.2.4.4.3.1.2.2.1.1
batch_cnt = 13

-> d = 13
selected_node
0.2.3.2.4.2.4.4.3.1.2.2.1.1.4
batch_cnt = 14

-> d = 14
selected_node
0.2.3.2.4.2.4.4.3.1.2.2.1.1.4.1
batch_cnt = 15

-> d = 15
selected_node
0.2.3.2.4.2.4.4.3.1.2.2.1.1.4.1.1
batch_cnt = 16

-> d = 16
selected_node
0.2.3.2.4.2.4.4.3.1.2.2.1.1.4.1.1.4
batch_cnt = 17

-> d = 1

nphases = 6
all_depths = [24, 1, 1, 1, 11, 12, 29]
ndepths_mean: 11.2857 (±4.3464)
nphases_mean: 10.9 (±1.6)
ndepths_mean: 7.0 (±1.2)


In [52]:
# print(config.cpuct)
# print(config.ds_alpha)
# print(config.ds_beta)
np.random.seed(100000+0)
random.seed(100000+0)
torch.manual_seed(100000+0)
torch.cuda.manual_seed(100000+0)

nphases_arr = []
ndepths_arr = []
for q_idx, question in enumerate(batch_of_questions):
    agent = test_nphases_mcts_search_v55.MCTS(config=config, question=question)
    agent_completions, nphases, ndepths = test_nphases_mcts_search_v55.mcts_search(question, agent, config, llm_vllm, llm_vllm_embeds, prm)
    # print(agent_completions)
    nphases_arr.append(nphases)
    ndepths_arr.append(ndepths)

nphases_mean = np.mean(nphases_arr)
nphases_std = np.std(nphases_arr, ddof=1)/np.sqrt(len(nphases_arr)) 
print(f"nphases_mean: {nphases_mean:0.1f} (\u00B1{nphases_std:0.1f})")

ndepths_mean = np.mean(ndepths_arr)
ndepths_std = np.std(ndepths_arr, ddof=1)/np.sqrt(len(ndepths_arr)) 
print(f"ndepths_mean: {ndepths_mean:0.1f} (\u00B1{ndepths_std:0.1f})")


-> p = 0
selected_node
0

-> d = 0
selected_node
0.4
batch_cnt = 1

-> d = 1
selected_node
0.4.3
batch_cnt = 2

-> d = 2
selected_node
0.4.3.1
batch_cnt = 3

-> d = 3
selected_node
0.4.3.1.4
batch_cnt = 4

-> d = 4
selected_node
0.4.3.1.4.4
batch_cnt = 5

-> d = 5
selected_node
0.4.3.1.4.4.2
batch_cnt = 6

-> d = 6
selected_node
None
batch_cnt = 7

-> d = 7

-> p = 1
selected_node
0.1

-> d = 0
selected_node
0.1.1
batch_cnt = 8

-> d = 1
selected_node
0.1.1.2
batch_cnt = 9

-> d = 2
selected_node
0.1.1.2.2
batch_cnt = 10

-> d = 3
selected_node
0.1.1.2.2.3
batch_cnt = 11

-> d = 4
selected_node
0.1.1.2.2.3.1
batch_cnt = 12

-> d = 5
selected_node
0.1.1.2.2.3.1.1
batch_cnt = 13

-> d = 6
selected_node
0.1.1.2.2.3.1.1.3
batch_cnt = 14

-> d = 7
selected_node
0.1.1.2.2.3.1.1.3.2
batch_cnt = 15

-> d = 8
selected_node
None
batch_cnt = 16

-> d = 9

-> p = 2
selected_node
0.1.1.3

-> d = 0
selected_node
0.1.1.3.3
batch_cnt = 17

-> d = 1
selected_node
0.1.1.3.3.4
batch_cnt = 18

-> d = 2
s

nphases = 16
all_depths = [7, 9, 5, 6, 5, 6, 1, 5, 7, 1, 1, 5, 5, 4, 4, 3, 5]
ndepths_mean: 4.6471 (±0.5351)


selected_node
0.1
batch_cnt = 1

-> d = 1
selected_node
0.1.3
batch_cnt = 2

-> d = 2
selected_node
0.1.3.3
batch_cnt = 3

-> d = 3
selected_node
0.1.3.3.4
batch_cnt = 4

-> d = 4
selected_node
0.1.3.3.4.2
batch_cnt = 5

-> d = 5
selected_node
0.1.3.3.4.2.2
batch_cnt = 6

-> d = 6
selected_node
0.1.3.3.4.2.2.4
batch_cnt = 7

-> d = 7
selected_node
0.1.3.3.4.2.2.4.4
batch_cnt = 8

-> d = 8
selected_node
0.1.3.3.4.2.2.4.4.4
batch_cnt = 9

-> d = 9
selected_node
0.1.3.3.4.2.2.4.4.4.2
batch_cnt = 10

-> d = 10
selected_node
0.1.3.3.4.2.2.4.4.4.2.3
batch_cnt = 11

-> d = 11
selected_node
0.1.3.3.4.2.2.4.4.4.2.3.3
batch_cnt = 12

-> d = 12
selected_node
0.1.3.3.4.2.2.4.4.4.2.3.3.1
batch_cnt = 13

-> d = 13
selected_node
0.1.3.3.4.2.2.4.4.4.2.3.3.1.4
batch_cnt = 14

-> d = 14
selected_node
0.1.3.3.4.2.2.4.4.4.2.3.3.1.4.3
batch_cnt = 15

-> d = 15
selected_node
0.1.3.3.4.2.2.4.4.4.2.3.3.1.4.3.4
batch_cnt = 16

-> d = 16
selected_node
0.1.3.3.4.2.2.4.4.4.2.3.3.1.4.3.4.2
batch_cnt = 17

-> d = 1

nphases = 6
all_depths = [20, 1, 1, 1, 14, 37, 5]
ndepths_mean: 11.2857 (±5.1210)


selected_node
0.1
batch_cnt = 1

-> d = 1
selected_node
0.1.4
batch_cnt = 2

-> d = 2
selected_node
None
batch_cnt = 3

-> d = 3

-> p = 1
selected_node
0.4

-> d = 0
selected_node
0.4.3
batch_cnt = 4

-> d = 1
selected_node
0.4.3.1
batch_cnt = 5

-> d = 2
selected_node
0.4.3.1.3
batch_cnt = 6

-> d = 3
selected_node
0.4.3.1.3.1
batch_cnt = 7

-> d = 4
selected_node
0.4.3.1.3.1.4
batch_cnt = 8

-> d = 5
selected_node
0.4.3.1.3.1.4.1
batch_cnt = 9

-> d = 6
selected_node
None
batch_cnt = 10

-> d = 7

-> p = 2
selected_node
0.3

-> d = 0
selected_node
0.3.2
batch_cnt = 11

-> d = 1
selected_node
0.3.2.3
batch_cnt = 12

-> d = 2
selected_node
None
batch_cnt = 13

-> d = 3

-> p = 3
selected_node
0.4.2

-> d = 0
selected_node
0.4.2.4
batch_cnt = 14

-> d = 1
selected_node
0.4.2.4.2
batch_cnt = 15

-> d = 2
selected_node
None
batch_cnt = 16

-> d = 3

-> p = 4
selected_node
0.2

-> d = 0
selected_node
0.2.3
batch_cnt = 17

-> d = 1
selected_node
0.2.3.1
batch_cnt = 18

-> d = 2
selected_no

...

nphases = 16
all_depths = [5, 4, 4, 3, 3, 4, 2, 2, 2, 1, 6, 2, 1, 4, 1, 0]
ndepths_mean: 2.7500 (±0.4133)


selected_node
0.4
batch_cnt = 1

-> d = 1
selected_node
0.4.1
batch_cnt = 2

-> d = 2
selected_node
0.4.1.2
batch_cnt = 3

-> d = 3
selected_node
None
batch_cnt = 4

-> d = 4

-> p = 1
selected_node
0.2

-> d = 0
selected_node
None
batch_cnt = 5

-> d = 1

-> p = 2
selected_node
0.1

-> d = 0
selected_node
0.1.2
batch_cnt = 6

-> d = 1
selected_node
0.1.2.3
batch_cnt = 7

-> d = 2
selected_node
0.1.2.3.3
batch_cnt = 8

-> d = 3
selected_node
0.1.2.3.3.3
batch_cnt = 9

-> d = 4
selected_node
0.1.2.3.3.3.3
batch_cnt = 10

-> d = 5
selected_node
0.1.2.3.3.3.3.1
batch_cnt = 11

-> d = 6
selected_node
None
batch_cnt = 12

-> d = 7

-> p = 3
selected_node
0.4.3

-> d = 0
selected_node
0.4.3.1
batch_cnt = 13

-> d = 1
selected_node
0.4.3.1.1
batch_cnt = 14

-> d = 2
selected_node
0.4.3.1.1.2
batch_cnt = 15

-> d = 3
selected_node
None
batch_cnt = 16

-> d = 4

-> p = 4
selected_node
0.3

-> d = 0
selected_node
0.3.1
batch_cnt = 17

-> d = 1
selected_node
0.3.1.2
batch_cnt = 18

-> d = 2
selec

nphases = 20
all_depths = [4, 1, 7, 4, 6, 3, 4, 1, 3, 5, 1, 1, 4, 3, 4, 2, 4, 1, 3, 0]
ndepths_mean: 3.0500 (±0.4134)


selected_node
0.2
batch_cnt = 1

-> d = 1
selected_node
0.2.3
batch_cnt = 2

-> d = 2
selected_node
0.2.3.2
batch_cnt = 3

-> d = 3
selected_node
0.2.3.2.4
batch_cnt = 4

-> d = 4
selected_node
0.2.3.2.4.2
batch_cnt = 5

-> d = 5
selected_node
0.2.3.2.4.2.4
batch_cnt = 6

-> d = 6
selected_node
0.2.3.2.4.2.4.4
batch_cnt = 7

-> d = 7
selected_node
0.2.3.2.4.2.4.4.3
batch_cnt = 8

-> d = 8
selected_node
0.2.3.2.4.2.4.4.3.1
batch_cnt = 9

-> d = 9
selected_node
0.2.3.2.4.2.4.4.3.1.2
batch_cnt = 10

-> d = 10
selected_node
0.2.3.2.4.2.4.4.3.1.2.2
batch_cnt = 11

-> d = 11
selected_node
0.2.3.2.4.2.4.4.3.1.2.2.1
batch_cnt = 12

-> d = 12
selected_node
0.2.3.2.4.2.4.4.3.1.2.2.1.1
batch_cnt = 13

-> d = 13
selected_node
0.2.3.2.4.2.4.4.3.1.2.2.1.1.4
batch_cnt = 14

-> d = 14
selected_node
0.2.3.2.4.2.4.4.3.1.2.2.1.1.4.1
batch_cnt = 15

-> d = 15
selected_node
0.2.3.2.4.2.4.4.3.1.2.2.1.1.4.1.1
batch_cnt = 16

-> d = 16
selected_node
0.2.3.2.4.2.4.4.3.1.2.2.1.1.4.1.1.4
batch_cnt = 17

-> d = 1

nphases = 6
all_depths = [24, 1, 1, 1, 11, 12, 29]
ndepths_mean: 11.2857 (±4.3464)
nphases_mean: 10.9 (±1.6)
ndepths_mean: 7.0 (±1.2)


In [51]:
# for idx, node in enumerate(agent_completions):
#     print(f"\n-> idx = {idx}")
#     print(node.state["text"])
# print(agent_completions)
arr = np.array([3, 7, 3, 3, 3, 3, 4, 2, 1, 0])
print(arr)
print(np.sum(arr))
arr_mean = np.mean(arr)
arr_std = np.std(arr, ddof=1)/np.sqrt(len(arr))
print(f"{arr_mean:0.4f} (\u00B1{arr_std:0.4f})")

[3 7 3 3 3 3 4 2 1 0]
29
2.9000 (±0.5859)
